In [ ]:
from datasets import load_dataset

ds = load_dataset("CardinalOperations/IndustryOR", split="test")

ds.to_json("industryor.jsonl")  


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 169.15ba/s]


114910

In [1]:
import json
from pathlib import Path

input_file = "industryor.jsonl"
output_dir = Path("questions_txt")
output_dir.mkdir(exist_ok=True)

with open(input_file, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        line = line.strip()
        if not line:
            continue

        obj = json.loads(line)

        file_id = obj.get("id", idx)
        content = obj.get("en_question", "").strip()

        out_path = output_dir / f"{file_id}.txt"
        with open(out_path, "w", encoding="utf-8") as out_f:
            out_f.write(content)

print(f"Done. Files saved in: {output_dir.resolve()}")

Done. Files saved in: /home/kongwoang/data/QACI/IndustryOR_Processing/questions_txt


In [1]:
import json
import csv
from pathlib import Path

input_file = "industryor.jsonl"
question_dir = Path("questions_txt")
question_dir.mkdir(exist_ok=True)

answers_csv = "answers.csv"

rows = []

with open(input_file, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        line = line.strip()
        if not line:
            continue

        obj = json.loads(line)

        sample_id = str(obj.get("id", idx)).strip().replace("/", "_")
        question = obj.get("en_question", "").strip()
        answer = obj.get("en_answer", "")
        difficulty = obj.get("difficulty", "")

        # Lưu question ra file txt
        txt_path = question_dir / f"{sample_id}.txt"
        with open(txt_path, "w", encoding="utf-8") as out_f:
            out_f.write(question)

        # Gom answer vào CSV
        rows.append({
            "id": sample_id,
            "en_answer": answer,
            "difficulty": difficulty,
        })

with open(answers_csv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "en_answer", "difficulty"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved questions to: {question_dir.resolve()}")
print(f"Saved answers CSV to: {Path(answers_csv).resolve()}")

Saved questions to: /home/kongwoang/data/QACI/IndustryOR_Processing/questions_txt
Saved answers CSV to: /home/kongwoang/data/QACI/IndustryOR_Processing/answers.csv


In [31]:
import json
import subprocess
import sys
from pathlib import Path

LLM_DIR = Path("LLM_output")

OUT_DIRS = {
    "input_format": Path("input_format"),
    "output_format": Path("output_format"),
    "example_input": Path("example_input"),
    "example_output": Path("example_output"),
    "run_logs": Path("run_logs"),
    "errors": Path("errors"),
}

for d in OUT_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)


def save_json(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def is_nonempty_file(path: Path) -> bool:
    return path.exists() and path.is_file() and path.stat().st_size > 0 and path.read_text(encoding="utf-8").strip() != ""


def run_python_file(py_file: Path):
    result = subprocess.run(
        [sys.executable, str(py_file)],
        capture_output=True,
        text=True,
        encoding="utf-8"
    )
    return result


def parse_solver_stdout(stdout_text: str):
    stdout_text = stdout_text.strip()
    if not stdout_text:
        raise ValueError("Python file produced empty stdout")

    try:
        return json.loads(stdout_text)
    except json.JSONDecodeError:
        start = stdout_text.find("{")
        end = stdout_text.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = stdout_text[start:end+1]
            return json.loads(candidate)
        raise ValueError("stdout is not valid JSON")


def process_one(sample_id: str):
    json_file = LLM_DIR / f"{sample_id}.json"
    py_file = LLM_DIR / f"{sample_id}.py"

    if not json_file.exists():
        raise FileNotFoundError(f"Missing JSON file: {json_file}")
    if not py_file.exists():
        raise FileNotFoundError(f"Missing Python file: {py_file}")

    if not is_nonempty_file(json_file):
        raise ValueError(f"JSON file is empty: {json_file}")

    if not is_nonempty_file(py_file):
        raise ValueError(f"Python file is empty: {py_file}")

    data = json.loads(json_file.read_text(encoding="utf-8"))

    required_keys = ["input_format", "output_format", "example_input"]
    missing = [k for k in required_keys if k not in data]
    if missing:
        raise KeyError(f"Missing keys in {json_file.name}: {missing}")

    save_json(data["input_format"], OUT_DIRS["input_format"] / f"{sample_id}.json")
    save_json(data["output_format"], OUT_DIRS["output_format"] / f"{sample_id}.json")
    save_json(data["example_input"], OUT_DIRS["example_input"] / f"{sample_id}.json")

    result = run_python_file(py_file)

    log_obj = {
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }
    save_json(log_obj, OUT_DIRS["run_logs"] / f"{sample_id}.json")

    if result.returncode != 0:
        raise RuntimeError(f"Python execution failed for {sample_id}")

    solver_output = parse_solver_stdout(result.stdout)

    if "example_output" not in solver_output:
        raise KeyError(f"'example_output' not found in stdout of {sample_id}.py")

    save_json(solver_output["example_output"], OUT_DIRS["example_output"] / f"{sample_id}.json")

    return {
        "sample_id": sample_id,
        "status": "ok",
        "solver_status": solver_output.get("status"),
        "objective_value": solver_output.get("objective_value"),
    }


def main():
    json_files = sorted(LLM_DIR.glob("*.json"))
    py_files = sorted(LLM_DIR.glob("*.py"))

    all_sample_ids = sorted(set([p.stem for p in json_files] + [p.stem for p in py_files]))

    results = []

    for sample_id in all_sample_ids:
        json_file = LLM_DIR / f"{sample_id}.json"
        py_file = LLM_DIR / f"{sample_id}.py"

        if not json_file.exists() or not py_file.exists():
            err = {
                "sample_id": sample_id,
                "status": "skipped",
                "reason": "missing_json_or_py_file",
            }
            save_json(err, OUT_DIRS["errors"] / f"{sample_id}.json")
            print(f"[skipped] {sample_id}: missing json or py")
            results.append(err)
            continue

        if not is_nonempty_file(json_file) or not is_nonempty_file(py_file):
            err = {
                "sample_id": sample_id,
                "status": "skipped",
                "reason": "empty_json_or_py_file",
            }
            save_json(err, OUT_DIRS["errors"] / f"{sample_id}.json")
            print(f"[skipped] {sample_id}: empty json or py")
            results.append(err)
            continue

        try:
            res = process_one(sample_id)
            print(f"[ok] {sample_id}")
            results.append(res)
        except Exception as e:
            err = {
                "sample_id": sample_id,
                "status": "error",
                "error": str(e),
            }
            save_json(err, OUT_DIRS["errors"] / f"{sample_id}.json")
            print(f"[error] {sample_id}: {e}")
            results.append(err)

    save_json(results, Path("process_summary.json"))
    print("Done.")


if __name__ == "__main__":
    main()

[ok] 1
[ok] 10
[ok] 100
[ok] 11
[ok] 12
[ok] 13
[ok] 14
[ok] 15
[ok] 16
[ok] 17
[ok] 18
[ok] 19
[ok] 2
[ok] 20
[ok] 21
[ok] 22
[ok] 23
[ok] 24
[ok] 25
[ok] 26
[ok] 27
[ok] 28
[ok] 29
[ok] 3
[ok] 30
[ok] 31
[ok] 32
[ok] 33
[ok] 34
[ok] 35
[ok] 36
[ok] 37
[ok] 38
[ok] 39
[ok] 4
[ok] 40
[ok] 41
[ok] 42
[ok] 43
[ok] 44
[ok] 45
[ok] 46
[ok] 47
[ok] 48
[ok] 49
[ok] 5
[ok] 50
[ok] 51
[ok] 52
[ok] 53
[ok] 54
[ok] 55
[ok] 56
[skipped] 57: empty json or py
[skipped] 58: empty json or py
[skipped] 59: empty json or py
[ok] 6
[skipped] 60: empty json or py
[skipped] 61: empty json or py
[skipped] 62: empty json or py
[skipped] 63: empty json or py
[skipped] 64: empty json or py
[skipped] 65: empty json or py
[skipped] 66: empty json or py
[skipped] 67: empty json or py
[skipped] 68: empty json or py
[skipped] 69: empty json or py
[ok] 7
[skipped] 70: empty json or py
[skipped] 71: empty json or py
[skipped] 72: empty json or py
[skipped] 73: empty json or py
[skipped] 74: empty json or py
[skipped

In [32]:
import json
import re
import math
import pandas as pd
from pathlib import Path

process_summary_path = Path("process_summary.json")
answers_csv_path = Path("answers.csv")
output_csv_path = Path("comparison.csv")

tolerance = 1e-6


def extract_first_number(text):
    if pd.isna(text):
        return None

    text = str(text).strip()
    if not text:
        return None

    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not matches:
        return None

    return float(matches[0])


def normalize_objective_value(x):
    if x is None:
        return None

    if isinstance(x, (int, float)) and not pd.isna(x):
        return float(x)

    if isinstance(x, str):
        s = x.strip()
        if not s:
            return None

        try:
            return float(s)
        except ValueError:
            pass

        matches = re.findall(r"-?\d+(?:\.\d+)?", s)
        if matches:
            return float(matches[0])
        return None

    if isinstance(x, dict):
        preferred_keys = [
            "objective_value",
            "value",
            "objective",
            "obj",
            "best_objective",
        ]
        for k in preferred_keys:
            if k in x:
                return normalize_objective_value(x[k])

        for v in x.values():
            y = normalize_objective_value(v)
            if y is not None:
                return y
        return None

    if isinstance(x, (list, tuple)):
        for v in x:
            y = normalize_objective_value(v)
            if y is not None:
                return y
        return None

    return None


def compare_values(obj_val, ans_val, tol=1e-6):
    obj_val = normalize_objective_value(obj_val)
    ans_val = normalize_objective_value(ans_val)

    if obj_val is None or ans_val is None:
        return None

    return math.isclose(obj_val, ans_val, rel_tol=tol, abs_tol=tol)


def numeric_sort_key(x):
    s = str(x).strip()
    m = re.search(r"\d+", s)
    if m:
        return (0, int(m.group()))
    return (1, s)


with open(process_summary_path, "r", encoding="utf-8") as f:
    process_summary = json.load(f)

summary_rows = []
for row in process_summary:
    status = row.get("status")

    # bỏ qua skipped
    if status != "ok":
        continue

    raw_obj = row.get("objective_value")
    summary_rows.append({
        "id": str(row.get("sample_id")),
        "status": status,
        "solver_status": row.get("solver_status"),
        "objective_value_raw": raw_obj,
        "objective_value": normalize_objective_value(raw_obj),
    })

df_summary = pd.DataFrame(summary_rows)

df_answers = pd.read_csv(answers_csv_path, dtype={"id": str})
df_answers["answer_numeric"] = df_answers["en_answer"].apply(extract_first_number)

df = df_summary.merge(
    df_answers[["id", "en_answer", "answer_numeric"]],
    on="id",
    how="left"
)

df["match"] = df.apply(
    lambda r: compare_values(r["objective_value"], r["answer_numeric"], tolerance),
    axis=1
)

df["abs_diff"] = df.apply(
    lambda r: None
    if r["objective_value"] is None or pd.isna(r["answer_numeric"])
    else abs(float(r["objective_value"]) - float(r["answer_numeric"])),
    axis=1
)

# sort theo số thật
df = df.sort_values(
    by="id",
    key=lambda col: col.map(numeric_sort_key)
).reset_index(drop=True)

df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

print("Saved to:", output_csv_path.resolve())
# print(df[["id", "status", "objective_value", "answer_numeric", "match", "abs_diff"]].head(30))
false_ids = df.loc[df["match"] == False, "id"].tolist()
print("False IDs:", false_ids)

Saved to: /home/kongwoang/data/QACI/IndustryOR_Processing/comparison.csv
False IDs: ['1', '29', '40', '47']
